## Model selection

### Three state-of-the-art algorithms
* Gradient Boosting Classifier
* Tensorflow Neural Network

In [1]:
import numpy as np
import pandas as pd
import random


from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras_tuner import RandomSearch


#set random seed
my_seed=42
np.random.seed(my_seed)

#diable warning printing
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Data set load
data=pd.read_csv('./data/C201905_counterfactuals/simulation_C201905_counterfactuals__ev0.1812.csv',index_col=0)


# Create a target variable, 'delay' and remove unnecessary columns
data['delay']=(data['actual_duration']>data['baseline_duration'])*1
data.drop(columns=['actual_duration','baseline_duration','critical_path'],inplace=True)
# Sort the columns
columns_sorted = sorted(data.columns, key=lambda x: int(x.replace('duration', '')) if 'duration' in x else float('inf'))
data = data[columns_sorted]
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration9,duration10,duration11,duration12,duration13,duration14,duration15,delay
0,1738.595774,1967.122966,256.504345,2186.408241,1723.181771,1898.648645,1795.289433,1599.760892,2224.382275,2489.048379,2124.279759,139.271436,8.498099,234.542254,7.655393,1
1,1717.864331,1881.352898,266.048082,2324.022291,1849.728508,2083.309016,1839.782891,1593.929687,2323.598219,2279.745524,2334.495021,145.448900,8.235859,238.908956,8.357302,1


In [3]:
# Normalize the dataset to improve the performance of the classifiers

def normalize_dataframe(df, feature_range=(0, 1)):
    """
    Normaliza un DataFrame de pandas utilizando MinMaxScaler.

    Args:
        df (pd.DataFrame): DataFrame a normalizar.
        feature_range (tuple): Rango de valores para la normalización (por defecto (0, 1)).

    Returns:
        pd.DataFrame: DataFrame normalizado.
        MinMaxScaler: Escalador utilizado para la normalización.
    """
    scaler = MinMaxScaler(feature_range=feature_range)
    normalized_data = scaler.fit_transform(df)
    normalized_df = pd.DataFrame(normalized_data, columns=df.columns, index=df.index)
    return normalized_df, scaler

def denormalize_dataframe(normalized_df, scaler):
    """
    Desnormaliza un DataFrame de pandas utilizando un MinMaxScaler previamente ajustado.

    Args:
        normalized_df (pd.DataFrame): DataFrame normalizado.
        scaler (MinMaxScaler): Escalador utilizado para la normalización.

    Returns:
        pd.DataFrame: DataFrame desnormalizado.
    """
    denormalized_data = scaler.inverse_transform(normalized_df)
    denormalized_df = pd.DataFrame(denormalized_data, columns=normalized_df.columns, index=normalized_df.index)
    return denormalized_df


data_to_normalize = data.iloc[:, :-1]  # Select all columns except the last one
normalized_data, my_scaler = normalize_dataframe(data_to_normalize)
data = pd.concat([normalized_data, data.iloc[:, -1]], axis=1)  # Combine normalized data with the last column

data.head(2)


,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration9,duration10,duration11,duration12,duration13,duration14,duration15,delay
0,0.751643,0.821171,0.653332,0.547362,0.393468,0.432562,0.715566,0.442461,0.804429,0.891496,0.551165,0.700622,0.641918,0.698465,0.380315,1
1,0.675055,0.594889,0.759338,0.713173,0.582989,0.679778,0.789338,0.432862,0.938094,0.637179,0.819892,0.830761,0.556337,0.752252,0.604439,1


In [4]:
# Split the dataset into features and class
X = data.drop(columns=['delay'])
y = data['delay']

# "stratify=y is used in train_test_split to ensure that the class distribution remains the same in both the training and test sets."
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=my_seed, stratify=y)

# Verify the split
print("Unique labels in training:", np.unique(y_train, return_counts=True))
print("Unique labels in test:", np.unique(y_test, return_counts=True))

Unique labels in training: (array([0, 1]), array([ 5564, 14436]))
Unique labels in test: (array([0, 1]), array([1391, 3609]))


### Gradient Boosting Clasifier

In [5]:
# Results dataframe
results = pd.DataFrame([],columns=['model','kf','Accuracy'])

# Stratified K-Fold cross-validation
kfold =StratifiedKFold(n_splits=5, random_state=my_seed,shuffle=True)

In [6]:
model='GradientBoostingC'
k=0
for train_index, test_index in kfold.split(X,y):

  # parameter optimization in the inner folds
    # n_estimators: number of  boosting stages
    # max_depth of the regression estimators
  mdc = GridSearchCV(GradientBoostingClassifier(),
                    param_grid={"n_estimators": np.linspace(50,200,10).astype('int'),  
                                "max_depth": np.linspace(1,10,5).astype('int')},
                    n_jobs=6)  
  mdc.fit(X.iloc[train_index,:], y.iloc[train_index])
  
  # score of the best model in the outer fold
  results = results._append( pd.DataFrame([[model,k, accuracy_score(y.iloc[test_index],mdc.best_estimator_.predict(X.iloc[test_index,:]))]],columns=['model','kf','Accuracy']))
  k+=1

### Tensorflow neural network

In [8]:
def build_model(hp):
    model = keras.Sequential([
        layers.Dense(hp.Int('units', min_value=32, max_value=128, step=32), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model='Tensorflow'
k=0
for train_index, test_index in kfold.split(X,y):

  # parameter optimization in the inner folds
    # units: number of  neurons in the hidden layer
    # learning_rate
  tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=1, directory='tuner_dir')

  tuner.search(X.iloc[train_index,:], y.iloc[train_index], epochs=30, validation_split=0.2, batch_size=32) 
  best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
  best_model = tuner.hypermodel.build(best_hps)
  best_model.fit(X.iloc[train_index,:], y.iloc[train_index], epochs=30, batch_size=32, validation_split=0.2, callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
  test_loss, test_acc = best_model.evaluate(X.iloc[test_index,:], y.iloc[test_index])
  
  # score of the best model in the outer fold
  results = results._append( pd.DataFrame([[model,k, test_acc]],columns=['model','kf','Accuracy']))
  k+=1

Reloading Tuner from tuner_dir/untitled_project/tuner0.json
Epoch 1/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 561us/step - accuracy: 0.7321 - loss: 0.6586 - val_accuracy: 0.8242 - val_loss: 0.4155
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 438us/step - accuracy: 0.8490 - loss: 0.3973 - val_accuracy: 0.8540 - val_loss: 0.3633
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8604 - loss: 0.3502 - val_accuracy: 0.8580 - val_loss: 0.3347
Epoch 4/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.8593 - loss: 0.3400 - val_accuracy: 0.8562 - val_loss: 0.3227
Epoch 5/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8656 - loss: 0.3187 - val_accuracy: 0.8635 - val_loss: 0.3227
Epoch 6/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 430us/step - accuracy: 0.8682 - loss: 0.3163 - val_accuracy: 0.8590 - val_loss: 0.3100
Epoch 7/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 431us/step - accuracy: 0.8669 - loss: 0.3099 - val_accuracy: 0.8558 - val_loss: 0.3111
Epoch 8/30
500/500 ━━━━━━━━━━━

### Comparison

In [9]:
results.groupby('model').agg({'Accuracy':['mean','std']}).sort_values(by=('Accuracy', 'mean'),ascending=False)


Accuracy          
                      mean       std
model                               
GradientBoostingC  0.98776  0.001108
Tensorflow         0.97664  0.007869

### Model hiperparameter tunning and model training

In [10]:
mdc = GridSearchCV(GradientBoostingClassifier(),
    param_grid={"n_estimators": np.linspace(50,200,10).astype('int'),  
                "max_depth": np.linspace(1,10,5).astype('int')},
    n_jobs=8) 
mdc.fit(X, y)
mdc.best_params_ # {'max_depth': 5, 'n_estimators': 100}

# Save the model
import joblib
joblib.dump(mdc.best_estimator_, 'gbc_best_model.pkl')
# Load the model
loaded_model = joblib.load('gbc_best_model.pkl')

In [11]:
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=1, directory='tuner_dir')
tuner.search(X, y, epochs=30, validation_split=0.2, batch_size=32)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Best TensorFlow Model: units={best_hps.get('units')}, learning_rate={best_hps.get('learning_rate')}")

# Save the model
best_model.save('tf_best_model.keras')
# Load the model
loaded_model = keras.models.load_model('tf_best_model.keras')

Reloading Tuner from tuner_dir/untitled_project/tuner0.json
Best TensorFlow Model: units=128, learning_rate=0.001
